In [11]:
import sys
import tkinter as tk
from tkinter import filedialog, messagebox, colorchooser

from PIL import Image, ImageTk, ImageDraw

sys.setrecursionlimit(100_000)

CANVAS_WIDTH = 700
CANVAS_HEIGHT = 450

BACKGROUND_COLOR = (255, 255, 255)
BOUNDARY_COLOR = (0, 0, 0)
fill_color = (255, 140, 40)
TRACE_COLOR = (255, 0, 0)

BRUSH_WIDTH = 3

In [12]:
def scanline_fill_recursive(image, seed_x, seed_y, painter):
    width, height = image.size

    if not (0 <= seed_x < width and 0 <= seed_y < height):
        return

    pixels = image.load()
    target_color = pixels[seed_x, seed_y]

    visited = bytearray(width * height)

    def index(x, y):
        return y * width + x

    def is_fillable(x, y):
        return (
            0 <= x < width
            and 0 <= y < height
            and not visited[index(x, y)]
            and pixels[x, y] == target_color
        )

    def fill_run(x, y):
        if not is_fillable(x, y):
            return

        left = x
        while left - 1 >= 0 and is_fillable(left - 1, y):
            left -= 1

        right = x
        while right + 1 < width and is_fillable(right + 1, y):
            right += 1

        for px in range(left, right + 1):
            visited[index(px, y)] = 1
            pixels[px, y] = painter(px, y)

        for neighbour_y in (y - 1, y + 1):
            if not (0 <= neighbour_y < height):
                continue

            px = left

            while px <= right:
                while px <= right and not is_fillable(px, neighbour_y):
                    px += 1

                if px > right:
                    break

                run_seed = px

                while px <= right and is_fillable(px, neighbour_y):
                    px += 1

                fill_run(run_seed, neighbour_y)

    fill_run(seed_x, seed_y)

In [13]:
def fill_with_color(image, x, y, selected_fill_color):
    if image.getpixel((x, y)) == BOUNDARY_COLOR:
        raise ValueError("Выбрана точка границы. Нужно щёлкнуть внутри области.")

    scanline_fill_recursive(
        image,
        x,
        y,
        painter=lambda _px, _py: selected_fill_color,
    )

In [14]:
def fill_with_pattern(image, x, y, pattern, repeat):
    if pattern is None:
        raise ValueError("Сначала нужно загрузить изображение для заливки.")

    if image.getpixel((x, y)) == BOUNDARY_COLOR:
        raise ValueError("Выбрана точка границы. Нужно щёлкнуть внутри области.")

    pattern_width, pattern_height = pattern.size
    pattern_pixels = pattern.load()

    if repeat:
        def painter(px, py):
            return pattern_pixels[
                px % pattern_width,
                py % pattern_height
            ]

    else:
        image_width, image_height = image.size

        if pattern_width < image_width or pattern_height < image_height:
            raise ValueError(
                "Для режима без повторения изображение должно быть "
                f"не меньше {image_width}×{image_height}. "
                f"Сейчас: {pattern_width}×{pattern_height}."
            )

        def painter(px, py):
            return pattern_pixels[px, py]

    scanline_fill_recursive(image, x, y, painter)

In [15]:
def same_color(a, b, tolerance=0):
    return all(
        abs(int(a[i]) - int(b[i])) <= tolerance
        for i in range(3)
    )


def trace_boundary(image, start_x, start_y):
    width, height = image.size
    pixels = image.load()

    boundary_color = pixels[start_x, start_y]
    start = (start_x, start_y)

    directions = [
        (1, 0),
        (1, 1),
        (0, 1),
        (-1, 1),
        (-1, 0),
        (-1, -1),
        (0, -1),
        (1, -1),
    ]

    def has_boundary_color(x, y):
        return (
            0 <= x < width
            and 0 <= y < height
            and same_color(pixels[x, y], boundary_color)
        )

    def touches_other_color(x, y):
        for dx, dy in directions:
            nx = x + dx
            ny = y + dy

            if not (0 <= nx < width and 0 <= ny < height):
                return True

            if not same_color(pixels[nx, ny], boundary_color):
                return True

        return False

    if not touches_other_color(start_x, start_y):
        raise ValueError(
            "Начальная точка находится внутри одноцветной области, "
            "а не на её границе."
        )

    contour = [start]
    current = start

    incoming_direction = None
    first_next = None

    max_steps = width * height * 4

    for _ in range(max_steps):
        search_start = (
            2
            if incoming_direction is None
            else (incoming_direction + 2) % 8
        )

        next_point = None
        next_direction = None

        for step in range(8):
            direction_index = (search_start - step) % 8
            dx, dy = directions[direction_index]

            nx = current[0] + dx
            ny = current[1] + dy

            if (
                has_boundary_color(nx, ny)
                and touches_other_color(nx, ny)
            ):
                next_point = (nx, ny)
                next_direction = direction_index
                break

        if next_point is None:
            break

        if first_next is None:
            first_next = next_point

        elif current == start and next_point == first_next:
            if contour and contour[-1] == start:
                contour.pop()
            break

        current = next_point
        incoming_direction = next_direction
        contour.append(current)

    else:
        raise RuntimeError(
            "Не удалось завершить обход границы: превышен лимит шагов."
        )

    if len(contour) < 2:
        raise ValueError(
            "Связный контур не найден. "
            "Проверьте, что граница непрерывна и имеет один цвет."
        )

    return contour


def draw_traced_boundary(image, contour, color=TRACE_COLOR):
    draw = ImageDraw.Draw(image)

    for x, y in contour:
        draw.point((x, y), fill=color)

In [16]:
class RasterAlgorithmsApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Растровые алгоритмы: 1а, 1б, 1в")

        self.image = Image.new(
            "RGB",
            (CANVAS_WIDTH, CANVAS_HEIGHT),
            BACKGROUND_COLOR,
        )

        self.pattern = None
        self.mode = "draw"
        self.last_mouse = None
        self.photo = None
        self.last_contour = []

        self._build_ui()
        self._refresh_canvas()

    def _build_ui(self):
        # Первая строка кнопок.
        toolbar = tk.Frame(
            self.root,
            padx=6,
            pady=6,
        )
        toolbar.pack(fill=tk.X)

        tk.Button(
            toolbar,
            text="Рисовать область",
            command=lambda: self._set_mode("draw"),
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar,
            text="1а: заливка цветом",
            command=lambda: self._set_mode("fill_color"),
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar,
            text="Выбрать цвет заливки",
            command=self._choose_fill_color,
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar,
            text="1б: текстура циклически",
            command=lambda: self._set_mode("fill_pattern_repeat"),
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar,
            text="1б: текстура без повтора",
            command=lambda: self._set_mode("fill_pattern_direct"),
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar,
            text="1в: обход границы",
            command=lambda: self._set_mode("trace"),
        ).pack(side=tk.LEFT, padx=2)

        toolbar2 = tk.Frame(
            self.root,
            padx=6,
            pady=0,
        )
        toolbar2.pack(fill=tk.X, pady=6)

        tk.Button(
            toolbar2,
            text="Загрузить текстуру",
            command=self._load_pattern,
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar2,
            text="Загрузить изображение",
            command=self._load_source_image,
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar2,
            text="Очистить",
            command=self._clear,
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            toolbar2,
            text="Сохранить",
            command=self._save_image,
        ).pack(side=tk.LEFT, padx=2)

        self.status = tk.StringVar(
            value="Режим: рисование границы мышкой"
        )

        tk.Label(
            toolbar2,
            textvariable=self.status,
            anchor="w",
        ).pack(side=tk.LEFT, padx=12)

        self.canvas = tk.Canvas(
            self.root,
            width=CANVAS_WIDTH,
            height=CANVAS_HEIGHT,
            bg="white",
            highlightthickness=1,
            highlightbackground="#777777",
            cursor="crosshair",
        )

        self.canvas.pack(
            padx=8,
            pady=8,
        )

        self.canvas.bind(
            "<Button-1>",
            self._on_left_press,
        )

        self.canvas.bind(
            "<B1-Motion>",
            self._on_left_drag,
        )

        self.canvas.bind(
            "<ButtonRelease-1>",
            self._on_left_release,
        )

    def _set_mode(self, mode):
        self.mode = mode
        self.last_mouse = None

        descriptions = {
            "draw":
                "Рисование: зажмите ЛКМ и нарисуйте замкнутую область.",

            "fill_color":
                "1а: щёлкните внутри области.",

            "fill_pattern_repeat":
                "1б: маленькая текстура будет повторяться циклически.",

            "fill_pattern_direct":
                "1б: большая текстура будет использована без масштабирования.",

            "trace":
                "1в: щёлкните по пикселу границы.",
        }

        self.status.set(descriptions[mode])

    def _on_left_press(self, event):
        x, y = self._clamp(
            event.x,
            event.y,
        )

        if self.mode == "draw":
            self.last_mouse = (x, y)
            self._draw_brush_point(x, y)
            return

        try:
            if self.mode == "fill_color":
                fill_with_color(
                    self.image,
                    x,
                    y,
                    fill_color,
                )

                self.status.set(
                    "1а выполнено: область залита цветом."
                )

            elif self.mode == "fill_pattern_repeat":
                fill_with_pattern(
                    self.image,
                    x,
                    y,
                    self.pattern,
                    repeat=True,
                )

                self.status.set(
                    "1б выполнено: текстура повторена циклически."
                )

            elif self.mode == "fill_pattern_direct":
                fill_with_pattern(
                    self.image,
                    x,
                    y,
                    self.pattern,
                    repeat=False,
                )

                self.status.set(
                    "1б выполнено: большая текстура использована напрямую."
                )

            elif self.mode == "trace":
                contour = trace_boundary(
                    self.image,
                    x,
                    y,
                )

                self.last_contour = contour

                draw_traced_boundary(
                    self.image,
                    contour,
                )

                self.status.set(
                    f"1в выполнено: найдено {len(contour)} точек границы."
                )

                print("\nКонтур (x, y) в порядке обхода:")

                for index, point in enumerate(contour):
                    print(
                        f"{index:5d}: {point}"
                    )

        except (ValueError, RuntimeError) as error:
            messagebox.showwarning(
                "Ошибка",
                str(error),
            )

        self._refresh_canvas()

    def _on_left_drag(self, event):
        if self.mode != "draw":
            return

        x, y = self._clamp(
            event.x,
            event.y,
        )

        if self.last_mouse is None:
            self.last_mouse = (x, y)

        draw = ImageDraw.Draw(self.image)

        draw.line(
            [
                self.last_mouse,
                (x, y),
            ],
            fill=BOUNDARY_COLOR,
            width=BRUSH_WIDTH,
        )

        self.last_mouse = (x, y)
        self._refresh_canvas()

    def _on_left_release(self, _event):
        self.last_mouse = None

    def _draw_brush_point(self, x, y):
        radius = max(
            1,
            BRUSH_WIDTH // 2,
        )

        draw = ImageDraw.Draw(self.image)

        draw.ellipse(
            (
                x - radius,
                y - radius,
                x + radius,
                y + radius,
            ),
            fill=BOUNDARY_COLOR,
        )

        self._refresh_canvas()

    def _choose_fill_color(self):
        global fill_color

        rgb, hex_color = colorchooser.askcolor(
            color=fill_color,
            title="Выберите цвет заливки",
            parent=self.root,
        )

        # Пользователь нажал Cancel.
        if rgb is None:
            return

        fill_color = tuple(
            int(round(component))
            for component in rgb
        )

        self.status.set(
            f"Цвет заливки выбран: {fill_color} ({hex_color})"
        )

    def _load_pattern(self):
        path = filedialog.askopenfilename(
            title="Выберите изображение для заливки",
            filetypes=[
                ("Изображения", "*.png *.jpg *.jpeg *.bmp"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg *.jpeg"),
                ("BMP", "*.bmp"),
                ("Все файлы", "*.*"),
            ],
        )

        if not path:
            return

        try:
            self.pattern = Image.open(path).convert("RGB")

            self.status.set(
                "Текстура загружена: "
                f"{self.pattern.width}×{self.pattern.height}"
            )

        except Exception as error:
            messagebox.showerror(
                "Ошибка",
                f"Не удалось загрузить изображение:\n{error}",
            )

    def _load_source_image(self):
        path = filedialog.askopenfilename(
            title="Загрузить исходное изображение",
            filetypes=[
                ("Изображения", "*.png *.jpg *.jpeg *.bmp"),
                ("PNG", "*.png"),
                ("JPEG", "*.jpg *.jpeg"),
                ("BMP", "*.bmp"),
                ("Все файлы", "*.*"),
            ],
        )

        if not path:
            return

        try:
            loaded = Image.open(path).convert("RGB")

            new_image = Image.new(
                "RGB",
                (CANVAS_WIDTH, CANVAS_HEIGHT),
                BACKGROUND_COLOR,
            )

            crop_width = min(
                loaded.width,
                CANVAS_WIDTH,
            )

            crop_height = min(
                loaded.height,
                CANVAS_HEIGHT,
            )

            crop = loaded.crop(
                (
                    0,
                    0,
                    crop_width,
                    crop_height,
                )
            )

            new_image.paste(
                crop,
                (0, 0),
            )

            self.image = new_image
            self.last_contour = []

            self._refresh_canvas()

            self.status.set(
                "Изображение загружено. "
                "Для 1в выберите режим и щёлкните по границе."
            )

        except Exception as error:
            messagebox.showerror(
                "Ошибка",
                f"Не удалось загрузить изображение:\n{error}",
            )

    def _save_image(self):
        path = filedialog.asksaveasfilename(
            title="Сохранить результат",
            defaultextension=".png",
            filetypes=[
                ("PNG", "*.png"),
                ("BMP", "*.bmp"),
                ("JPEG", "*.jpg *.jpeg"),
            ],
        )

        if not path:
            return

        try:
            self.image.save(path)

            self.status.set(
                f"Результат сохранён: {path}"
            )

        except Exception as error:
            messagebox.showerror(
                "Ошибка",
                f"Не удалось сохранить изображение:\n{error}",
            )

    def _clear(self):
        self.image = Image.new(
            "RGB",
            (CANVAS_WIDTH, CANVAS_HEIGHT),
            BACKGROUND_COLOR,
        )

        self.last_contour = []
        self._refresh_canvas()
        self._set_mode("draw")

    @staticmethod
    def _clamp(x, y):
        return (
            max(
                0,
                min(
                    CANVAS_WIDTH - 1,
                    int(x),
                ),
            ),
            max(
                0,
                min(
                    CANVAS_HEIGHT - 1,
                    int(y),
                ),
            ),
        )

    def _refresh_canvas(self):
        self.photo = ImageTk.PhotoImage(
            self.image
        )

        self.canvas.delete("all")

        self.canvas.create_image(
            0,
            0,
            anchor=tk.NW,
            image=self.photo,
        )

In [17]:
try:
    if "root" in globals() and root.winfo_exists():
        root.destroy()
except Exception:
    pass

try:
    ipython = get_ipython()
except NameError:
    ipython = None

if ipython is not None:
    ipython.run_line_magic("gui", "tk")

root = tk.Tk()
app = RasterAlgorithmsApp(root)

print("Окно программы запущено.")

Окно программы запущено.
